Library

In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
import numpy as np
import cv2
from scipy.fftpack import dct
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# Class Novelty
class ChrominanceFrequencyTransform:
    def __call__(self, img):
        img_np = np.array(img)
        if len(img_np.shape) < 3: img_np = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)
        img_ycc = cv2.cvtColor(img_np, cv2.COLOR_RGB2YCrCb)
        chroma_1 = img_ycc[:,:,1]
        chroma_2 = img_ycc[:,:,2]
        dct_1 = dct(dct(chroma_1.T, norm='ortho').T, norm='ortho')
        dct_2 = dct(dct(chroma_2.T, norm='ortho').T, norm='ortho')
        dct_combined = (np.log(np.abs(dct_1) + 1e-12) + np.log(np.abs(dct_2) + 1e-12)) / 2
        dct_norm = (dct_combined - np.min(dct_combined)) / (np.max(dct_combined) - np.min(dct_combined))
        dct_stacked = np.stack([dct_norm] * 3, axis=2)
        return dct_stacked.astype(np.float32)

    # Helper untuk visualisasi
    def get_visual(self, img):
        res = self.__call__(img)
        return res[:,:,0]

In [ ]:
print("⬆️ Upload file 'my_deepfake_model.pth'...")
uploaded_model = files.upload()

if 'my_deepfake_model.pth' in uploaded_model:
    print("✅ Model berhasil diupload!")
else:
    print("⚠️ Peringatan: Nama file mungkin berbeda. Pastikan namanya benar.")

⬆️ Upload file 'my_deepfake_model.pth'...


In [ ]:
# 1. Load Arsitektur & Bobot
model = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
try:
    model.load_state_dict(torch.load("model_deepfake_novelty.pth", map_location=device))
    model.to(device)
    model.eval()
    print("🤖 Model siap digunakan!")
except:
    print("❌ Gagal load model. Pastikan langkah Upload Model sudah dijalankan.")

# 2. Upload Foto Test
print("\n⬆️ Upload Images untuk dites...")
uploaded_img = files.upload()

# Label mapping (Sesuaikan dengan output 'Kelas ditemukan' di notebook training)
# Biasanya urutan abjad folder dataset: 0=Fake, 1=Real (atau sebaliknya, cek training!)
LABELS = {0: 'Deepfake / Palsu', 1: 'Real / Asli'}

for fn in uploaded_img.keys():
    path = fn
    img = Image.open(path).convert('RGB')
    img_resized = img.resize((224, 224))

    # Transform
    transformer = ChrominanceFrequencyTransform()
    visual_dct = transformer.get_visual(img_resized)

    input_tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])(transformer(img_resized)).unsqueeze(0).to(device)

    # Prediksi
    with torch.no_grad():
        out = model(input_tensor)
        probs = torch.nn.functional.softmax(out, dim=1)
        conf, pred = torch.max(probs, 1)

    result_text = LABELS.get(pred.item(), "Unknown")

    # Plotting
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(img_resized)
    ax[0].set_title("Foto Input")
    ax[0].axis('off')

    ax[1].imshow(visual_dct, cmap='inferno')
    ax[1].set_title("Analisis Krominansi (AI View)")
    ax[1].axis('off')

    plt.suptitle(f"Hasil: {result_text} ({conf.item()*100:.1f}%)", fontsize=14, fontweight='bold', color='blue')
    plt.show()